# DataSphere scale runs: NMSQA, 4 configurations

Repository: `/home/jupyter/project/slm-audio-evidence`  
Storage root: `/home/jupyter/filestore/space1/slm-audio-evidence`  
Upload: `<storage root>/input/scale_nmsqa_datasphere.zip`

In [ ]:
!nvidia-smi -L

In [ ]:
# Persistent storage paths.
from pathlib import Path
import os

FILESTORE = Path("/home/jupyter/filestore/space1")
if not FILESTORE.is_dir():
    raise RuntimeError(
        f"Storage mount not found: {FILESTORE}"
    )

STORAGE_ROOT = FILESTORE / "slm-audio-evidence"
INPUT_DIR = STORAGE_ROOT / "input"
HF_HOME = STORAGE_ROOT / "huggingface"
HF_HUB_CACHE = HF_HOME / "hub"
PYTHON_DEPS = STORAGE_ROOT / "python"
PIP_CACHE_DIR = STORAGE_ROOT / "pip_cache"
RUNS_DIR = STORAGE_ROOT / "results" / "scale_nmsqa"
SCALE_DATA = STORAGE_ROOT / "scale_data"

for path in (INPUT_DIR, HF_HUB_CACHE, PYTHON_DEPS, PIP_CACHE_DIR, RUNS_DIR, SCALE_DATA):
    path.mkdir(parents=True, exist_ok=True)

# Used by shell commands below.
os.environ.update({
    "INPUT_DIR": str(INPUT_DIR),
    "HF_HOME": str(HF_HOME),
    "HF_HUB_CACHE": str(HF_HUB_CACHE),
    "PYTHON_DEPS": str(PYTHON_DEPS),
    "PIP_CACHE_DIR": str(PIP_CACHE_DIR),
    "RUNS_DIR": str(RUNS_DIR),
    "SCALE_DATA": str(SCALE_DATA),
})

probe = STORAGE_ROOT / ".write_test"
probe.write_text("ok", encoding="utf-8")
probe.unlink()
print(f"Writable storage: {STORAGE_ROOT}")

In [ ]:
# Repository: /home/jupyter/project/slm-audio-evidence
%cd /home/jupyter/project
!test -d slm-audio-evidence/.git || git clone https://github.com/ladnlav/slm-audio-evidence.git
!git -C slm-audio-evidence pull --ff-only
%cd slm-audio-evidence

In [ ]:
# Python packages: <storage>/slm-audio-evidence/python
from pathlib import Path
import os
import subprocess
import sys

deps = Path(os.environ["PYTHON_DEPS"])
marker = deps / ".scale_deps_v2_ready"
packages = [
    "transformers==4.45.2",
    "tokenizers==0.20.3",
    "huggingface-hub==0.25.2",
    "safetensors==0.4.5",
    "accelerate==0.34.2",
    "bitsandbytes==0.43.3",
    "librosa==0.10.2.post1",
    "soundfile==0.12.1",
    "soxr==0.3.7",
    "lazy-loader==0.4",
]

if not marker.is_file():
    install_env = os.environ.copy()
    install_env.update({
        "PIP_DISABLE_PIP_VERSION_CHECK": "1",
        "PYTHONNOUSERSITE": "1",
    })
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--no-deps", "--target", str(deps), *packages],
        check=True,
        env=install_env,
    )
    marker.touch()

deps_str = str(deps)
if deps_str not in sys.path:
    sys.path.insert(0, deps_str)

import torch
import torchaudio
import transformers

print(torch.__version__, torchaudio.__version__, transformers.__version__)

In [ ]:
# Extract the validated scale bundle from persistent storage.
from pathlib import Path
import hashlib
import os
import zipfile

scale_data = Path(os.environ["SCALE_DATA"])
archive = Path(os.environ["INPUT_DIR"]) / "scale_nmsqa_datasphere.zip"
stored_audio = scale_data / "data" / "raw" / "nmsqa_squad_30s"
stored_manifest = scale_data / "data" / "manifests" / "scale_nmsqa.jsonl"
if len(list(stored_audio.glob("*.wav"))) != 51:
    if not archive.is_file():
        raise FileNotFoundError(f"Upload the scale bundle to: {archive}")
    with zipfile.ZipFile(archive) as bundle:
        bad_member = bundle.testzip()
        if bad_member is not None:
            raise ValueError(f"Corrupt ZIP member: {bad_member}")
        bundle.extractall(scale_data)

repo_manifest = Path.cwd() / "data" / "manifests" / "scale_nmsqa.jsonl"
if not stored_manifest.is_file():
    raise FileNotFoundError(f"Bundled manifest not found: {stored_manifest}")
repo_hash = hashlib.sha256(repo_manifest.read_bytes()).hexdigest()
stored_hash = hashlib.sha256(stored_manifest.read_bytes()).hexdigest()
if repo_hash != stored_hash:
    raise ValueError("The uploaded ZIP and repository manifest differ. Rebuild the ZIP.")

repo_audio = Path.cwd() / "data" / "raw" / "nmsqa_squad_30s"
repo_audio.parent.mkdir(parents=True, exist_ok=True)
if repo_audio.is_symlink() and repo_audio.resolve() != stored_audio.resolve():
    repo_audio.unlink()
if not repo_audio.exists():
    repo_audio.symlink_to(stored_audio, target_is_directory=True)

wav_count = len(list(repo_audio.glob("*.wav")))
if wav_count != 51:
    raise RuntimeError(f"Expected 51 WAV files, found {wav_count} in {repo_audio}")
print(f"Audio ready: {wav_count} files -> {stored_audio}")

In [ ]:
# Model cache: <storage>/slm-audio-evidence/huggingface
import os
import subprocess
import sys

run_env = os.environ.copy()
run_env.pop("TRANSFORMERS_CACHE", None)
run_env["PYTHONNOUSERSITE"] = "1"
run_env["PYTHONPATH"] = os.pathsep.join(
    filter(None, [os.environ["PYTHON_DEPS"], run_env.get("PYTHONPATH")])
)

runs = [
    ("qwen2audio", "plain"),
    ("qwen2audio", "s1_idk"),
    ("cascade", "plain"),
    ("cascade", "s1_idk"),
]
for model, strategy in runs:
    print(f"\n=== {model} x {strategy} ===", flush=True)
    subprocess.run(
        [
            sys.executable,
            "-m", "src.inference",
            "--model", model,
            "--strategy", strategy,
            "--data", "data/manifests/scale_nmsqa.jsonl",
            "--out", os.environ["RUNS_DIR"],
        ],
        check=True,
        env=run_env,
    )

In [ ]:
# Results: <storage>/slm-audio-evidence/results/scale_nmsqa
from pathlib import Path
import os

runs = [
    ("qwen2audio", "plain"),
    ("qwen2audio", "s1_idk"),
    ("cascade", "plain"),
    ("cascade", "s1_idk"),
]
for model, strategy in runs:
    response_files = sorted(
        Path(os.environ["RUNS_DIR"]).glob(
            f"{model}_{strategy}_*/responses.jsonl"
        )
    )
    if not response_files:
        raise FileNotFoundError(f"Missing responses for {model} x {strategy}")
    responses = response_files[-1]
    completed = sum(
        1
        for line in responses.read_text(encoding="utf-8").splitlines()
        if line.strip()
    )
    assert completed == 380, (
        f"Expected 380 responses, got {completed}: {responses}"
    )
    print(f"Complete: {model} x {strategy}: {completed}/380 -> {responses}")